In [ ]:
import boto3
import sagemaker

In [ ]:
#check role


role = sagemaker.get_execution_role()

print(role)

In [ ]:
#check your bucket


s3 = boto3.client("s3")

response = s3.list_buckets()

for bucket in response['Buckets']:
    print(bucket['Name'])

In [ ]:
#package up a trained model to deploy it  AWS SageMaker
#because deploying our model to cloud platforms like AWS SageMaker, Google Cloud Vertex AI, or Azure ML, 
#their systems are hardcoded to look for a single compressed archive.

import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "model/model_churn.joblib",
        arcname="model_churn.joblib"
    )

In [ ]:
#Upload model to S3, because deployment will take the model form s3

s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-<account-id>"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [ ]:
import subprocess
subprocess.run(["pip", "install", "joblib", "scikit-learn", "numpy", "pandas"], check=True)

In [ ]:
#check model can be run or not

from inference import model_fn

model = model_fn("model")

In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel


# ---- EDIT THESE ---------------------------------------------------------
BUCKET = "sagemaker-us-east-1-<account-id>"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "churn-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )
    
    print("Testing endpoint with sample data...")

    sample = {
        "instances": [[30, "Female", 5, 10, 2, 3, "Standard", "Annual", 500, 10]]}
    
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=str(sample).replace("'", '"'),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))


if __name__ == "__main__":
    main()


In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logsV2:log-groups

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "churn-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

# 1. Delete all endpoints
print("=== Endpoints ===")
for ep in sm.list_endpoints()['Endpoints']:
    sm.delete_endpoint(EndpointName=ep['EndpointName'])
    print(f"Deleted endpoint: {ep['EndpointName']}")

# 2. Delete all endpoint configs
print("=== Endpoint Configs ===")
for ec in sm.list_endpoint_configs()['EndpointConfigs']:
    sm.delete_endpoint_config(EndpointConfigName=ec['EndpointConfigName'])
    print(f"Deleted config: {ec['EndpointConfigName']}")

# 3. Delete all models
print("=== Models ===")
for m in sm.list_models()['Models']:
    sm.delete_model(ModelName=m['ModelName'])
    print(f"Deleted model: {m['ModelName']}")

print("\nAll SageMaker resources cleared!")
print("Don't forget to manually check S3 buckets and EBS volumes in the console.")

In [ ]:
import boto3
sm = boto3.client("sagemaker", region_name="us-east-1")

print("ENDPOINTS:", sm.list_endpoints()['Endpoints'])
print("CONFIGS:", sm.list_endpoint_configs()['EndpointConfigs'])
print("MODELS:", sm.list_models()['Models'])

In [ ]:
print("TRAINING JOBS:", sm.list_training_jobs(StatusEquals='InProgress')['TrainingJobSummaries'])